In [ ]:
# =========================================================
# BASELINE DiD + SUN-ABRAHAM-STYLE EVENT STUDY
# Single notebook cell
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from IPython.display import display

# ---------------------------------------------------------
# 0. Plot style
# ---------------------------------------------------------
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parents[1]
PATH = PROJECT_ROOT / "data" / "processed" / "for_analysis" / "va_did_panel.csv"

df = pd.read_csv(PATH)
df.columns = [c.strip() for c in df.columns]

# statsmodels-friendly types
df["census_geoid"] = df["census_geoid"].astype(str)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
df["first_entry_year"] = pd.to_numeric(df["first_entry_year"], errors="coerce").astype(float)
df["treated"] = pd.to_numeric(df["treated"], errors="coerce").fillna(0).astype(int)
df["weight"] = pd.to_numeric(df["weight"], errors="coerce").astype(float)

numeric_cols = [
    "avg_total_lmp", "avg_energy", "avg_congestion", "avg_loss",
    "n_pnodes", "inefficient_pricing", "event_time"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

expected = [
    "census_geoid", "year", "avg_total_lmp", "avg_energy",
    "avg_congestion", "avg_loss", "n_pnodes", "inefficient_pricing",
    "first_entry_year", "treated", "event_time", "weight", "group"
]
missing = [c for c in expected if c not in df.columns]
print("Missing columns:", missing)

# Treatment variables
df["ever_treated"] = (
    df["first_entry_year"].notna() & (df["first_entry_year"] > 0)
).astype(int)

df["post"] = (
    (df["ever_treated"] == 1) & (df["year"] >= df["first_entry_year"])
).astype(int)

df["event_time_check"] = np.where(
    df["ever_treated"] == 1,
    df["year"] - df["first_entry_year"],
    np.nan
)

# Outcome validation
df["ineff_check"] = df["avg_congestion"] + df["avg_loss"]

# Panel checks
dupes = df.duplicated(subset=["census_geoid", "year"]).sum()
print(f"Duplicate tract-year rows: {dupes}")
print("Max |inefficient_pricing - (avg_congestion + avg_loss)| =",
      np.nanmax(np.abs(df["inefficient_pricing"] - df["ineff_check"])))

print("\nRows:", len(df))
print("Tracts:", df["census_geoid"].nunique())
print("Years:", sorted(df["year"].dropna().unique().tolist()))
print("Ever-treated tracts:", df.loc[df["ever_treated"] == 1, "census_geoid"].nunique())
print("Never-treated tracts:", df.loc[df["ever_treated"] == 0, "census_geoid"].nunique())

print("\nTreated cohorts:")
display(
    df.loc[df["ever_treated"] == 1, ["census_geoid", "first_entry_year"]]
      .drop_duplicates()
      .sort_values(["first_entry_year", "census_geoid"])
)

# ---------------------------------------------------------
# 2. Pre-treatment sample for balance / trend checks
# ---------------------------------------------------------
pre = df.loc[
    (df["ever_treated"] == 0) | (df["year"] < df["first_entry_year"])
].copy()

pre["census_geoid"] = pre["census_geoid"].astype(str)
pre["year"] = pre["year"].astype(int)
pre["ever_treated"] = pre["ever_treated"].astype(int)
pre["weight"] = pre["weight"].astype(float)
pre["inefficient_pricing"] = pre["inefficient_pricing"].astype(float)

print("\nPre-treatment rows:", len(pre))
print("Pre-treatment treated tracts:", pre.loc[pre["ever_treated"] == 1, "census_geoid"].nunique())
print("Pre-treatment control tracts:", pre.loc[pre["ever_treated"] == 0, "census_geoid"].nunique())

# ---------------------------------------------------------
# 3. Pre-treatment balance table
# ---------------------------------------------------------
balance_vars = [
    "inefficient_pricing",
    "avg_congestion",
    "avg_loss",
    "avg_total_lmp",
    "avg_energy",
    "n_pnodes"
]

def weighted_mean(x, w):
    m = np.isfinite(x) & np.isfinite(w)
    x = np.asarray(x[m], dtype=float)
    w = np.asarray(w[m], dtype=float)
    return np.sum(w * x) / np.sum(w)

def weighted_var(x, w):
    m = np.isfinite(x) & np.isfinite(w)
    x = np.asarray(x[m], dtype=float)
    w = np.asarray(w[m], dtype=float)
    mu = np.sum(w * x) / np.sum(w)
    return np.sum(w * (x - mu) ** 2) / np.sum(w)

def smd(x_t, w_t, x_c, w_c):
    mt = weighted_mean(x_t, w_t)
    mc = weighted_mean(x_c, w_c)
    vt = weighted_var(x_t, w_t)
    vc = weighted_var(x_c, w_c)
    return (mt - mc) / np.sqrt((vt + vc) / 2)

balance_rows = []
for v in balance_vars:
    tm = pre["ever_treated"] == 1
    cm = pre["ever_treated"] == 0

    mt = weighted_mean(pre.loc[tm, v], pre.loc[tm, "weight"])
    mc = weighted_mean(pre.loc[cm, v], pre.loc[cm, "weight"])
    balance_rows.append({
        "variable": v,
        "treated_pre_mean": mt,
        "control_pre_mean": mc,
        "difference": mt - mc,
        "std_mean_diff": smd(
            pre.loc[tm, v], pre.loc[tm, "weight"],
            pre.loc[cm, v], pre.loc[cm, "weight"]
        )
    })

balance_table = pd.DataFrame(balance_rows)
print("\nPre-treatment balance table")
display(balance_table.round(4))

# tract-level t-tests on pre means
tract_pre = (
    pre.groupby(["census_geoid", "ever_treated"], as_index=False)
       .agg(
           weight=("weight", "mean"),
           inefficient_pricing=("inefficient_pricing", "mean"),
           avg_congestion=("avg_congestion", "mean"),
           avg_loss=("avg_loss", "mean"),
           avg_total_lmp=("avg_total_lmp", "mean"),
           avg_energy=("avg_energy", "mean"),
           n_pnodes=("n_pnodes", "mean")
       )
)

ttest_rows = []
for v in balance_vars:
    x1 = tract_pre.loc[tract_pre["ever_treated"] == 1, v].dropna()
    x0 = tract_pre.loc[tract_pre["ever_treated"] == 0, v].dropna()
    tstat, pval = stats.ttest_ind(x1, x0, equal_var=False)
    ttest_rows.append({"variable": v, "t_stat": tstat, "p_value": pval})

balance_ttests = pd.DataFrame(ttest_rows)
print("\nPre-treatment t-tests")
display(balance_ttests.round(4))

# ---------------------------------------------------------
# 4. Raw pre-trend plot
# ---------------------------------------------------------
year_group = (
    pre.groupby(["year", "ever_treated"])
       .apply(lambda g: np.average(g["inefficient_pricing"], weights=g["weight"]))
       .reset_index(name="wmean_inefficient_pricing")
)

pivot_pre = year_group.pivot(index="year", columns="ever_treated", values="wmean_inefficient_pricing")
pivot_pre.columns = ["never_treated", "ever_treated"]

ax = pivot_pre.plot(marker="o")
ax.set_title("Pre-treatment trends in inefficient pricing")
ax.set_ylabel("Weighted mean inefficient pricing")
ax.set_xlabel("Year")
plt.legend(["Never treated", "Ever treated"])
plt.show()

# ---------------------------------------------------------
# 5. Formal pre-trend regression
# ---------------------------------------------------------
pre_years = sorted(pre["year"].dropna().unique().tolist())
ref_year = pre_years[0]
print("Reference pre-period year:", ref_year)

pretrend_formula = (
    f"inefficient_pricing ~ C(census_geoid) + C(year) "
    f"+ C(year, Treatment(reference={ref_year})):ever_treated"
)

pretrend_mod = smf.wls(
    formula=pretrend_formula,
    data=pre,
    weights=pre["weight"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": pre["census_geoid"]}
)

print("\nPre-trend regression summary")
print(pretrend_mod.summary())

interaction_terms = [
    p for p in pretrend_mod.params.index
    if "C(year, Treatment" in p and ":ever_treated" in p
]
if interaction_terms:
    restriction = " = 0, ".join(interaction_terms) + " = 0"
    print("\nJoint pre-trend test:")
    print(pretrend_mod.f_test(restriction))

# ---------------------------------------------------------
# 6. Baseline TWFE DiD
# ---------------------------------------------------------
did_formula = "inefficient_pricing ~ post + C(census_geoid) + C(year)"
did_mod = smf.wls(
    formula=did_formula,
    data=df,
    weights=df["weight"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["census_geoid"]}
)

print("\nBaseline TWFE DiD")
print(did_mod.summary())

# Multiple outcomes
outcomes = ["inefficient_pricing", "avg_congestion", "avg_loss", "avg_total_lmp", "avg_energy"]
did_rows = []
for y in outcomes:
    mod = smf.wls(
        formula=f"{y} ~ post + C(census_geoid) + C(year)",
        data=df,
        weights=df["weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["census_geoid"]}
    )
    did_rows.append({
        "outcome": y,
        "coef_post": mod.params["post"],
        "se": mod.bse["post"],
        "p_value": mod.pvalues["post"]
    })

did_outcome_results = pd.DataFrame(did_rows)
print("\nTWFE DiD across outcomes")
display(did_outcome_results.round(4))

# ---------------------------------------------------------
# 7. Sun-Abraham-style interaction-weighted event study
# ---------------------------------------------------------
def run_sunab_style(data, outcome, min_e=-4, max_e=5, ref_e=-1):
    d = data.copy()

    # cohorts
    cohorts = sorted(
        d.loc[d["ever_treated"] == 1, "first_entry_year"]
         .dropna().unique().astype(int).tolist()
    )

    # bin event time
    d["event_time_binned"] = d["event_time_check"]
    d.loc[d["event_time_binned"] < min_e, "event_time_binned"] = min_e
    d.loc[d["event_time_binned"] > max_e, "event_time_binned"] = max_e

    def safe_event_name(e):
        return f"m{abs(e)}" if e < 0 else f"p{e}"

    # cohort x event-time dummies
    interaction_cols = []
    for g in cohorts:
        for e in range(min_e, max_e + 1):
            if e == ref_e:
                continue
            col = f"g{g}_e{safe_event_name(e)}"
            d[col] = (
                (d["ever_treated"] == 1) &
                (d["first_entry_year"] == g) &
                (d["event_time_binned"] == e)
            ).astype(int)
            interaction_cols.append(col)

    rhs = " + ".join(interaction_cols)
    formula = f"{outcome} ~ {rhs} + C(census_geoid) + C(year)"

    mod = smf.wls(
        formula=formula,
        data=d,
        weights=d["weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["census_geoid"]}
    )

    # cohort-size weights
    cohort_sizes = (
        d.loc[d["ever_treated"] == 1, ["census_geoid", "first_entry_year"]]
         .drop_duplicates()
         .groupby("first_entry_year")
         .size()
         .rename("n_cohort")
         .reset_index()
    )
    cohort_sizes["cohort_weight"] = cohort_sizes["n_cohort"] / cohort_sizes["n_cohort"].sum()
    cohort_weight_map = dict(zip(cohort_sizes["first_entry_year"].astype(int), cohort_sizes["cohort_weight"]))

    # aggregate across cohorts by event time
    rows = []
    for e in range(min_e, max_e + 1):
        if e == ref_e:
            continue

        relevant_terms = []
        relevant_weights = []

        for g in cohorts:
            term = f"g{g}_e{safe_event_name(e)}"
            if term in mod.params.index:
                relevant_terms.append(term)
                relevant_weights.append(cohort_weight_map[g])

        if len(relevant_terms) == 0:
            continue

        relevant_weights = np.array(relevant_weights, dtype=float)
        relevant_weights = relevant_weights / relevant_weights.sum()

        beta = sum(w * mod.params[t] for w, t in zip(relevant_weights, relevant_terms))
        V = mod.cov_params().loc[relevant_terms, relevant_terms].values
        var = relevant_weights @ V @ relevant_weights
        se = np.sqrt(var)

        rows.append({
            "event_time": e,
            "coef": beta,
            "se": se,
            "ci_low": beta - 1.96 * se,
            "ci_high": beta + 1.96 * se,
            "n_cohorts_used": len(relevant_terms)
        })

    event_df = pd.DataFrame(rows).sort_values("event_time")

    # joint lead test on underlying cohort-specific lead coefficients
    lead_terms = []
    for g in cohorts:
        for e in [-4, -3, -2]:
            term = f"g{g}_e{safe_event_name(e)}"
            if term in mod.params.index:
                lead_terms.append(term)

    lead_test = None
    if lead_terms:
        restriction = " = 0, ".join(lead_terms) + " = 0"
        lead_test = mod.f_test(restriction)

    # overall post-treatment ATT
    post_event_df = event_df.loc[event_df["event_time"] >= 0].copy()
    overall_att = np.nan
    overall_se = np.nan
    overall_ci = (np.nan, np.nan)

    if len(post_event_df) > 0:
        post_terms = []
        post_weights = []

        for e in post_event_df["event_time"]:
            for g in cohorts:
                term = f"g{g}_e{safe_event_name(int(e))}"
                if term in mod.params.index:
                    post_terms.append(term)
                    post_weights.append((1 / len(post_event_df)) * cohort_weight_map[g])

        if len(post_terms) > 0:
            post_weights = np.array(post_weights, dtype=float)
            V_post = mod.cov_params().loc[post_terms, post_terms].values
            overall_att = post_weights @ mod.params[post_terms].values
            overall_var = post_weights @ V_post @ post_weights
            overall_se = np.sqrt(overall_var)
            overall_ci = (overall_att - 1.96 * overall_se, overall_att + 1.96 * overall_se)

    return {
        "model": mod,
        "event_df": event_df,
        "lead_test": lead_test,
        "overall_att": overall_att,
        "overall_se": overall_se,
        "overall_ci": overall_ci
    }

# Run preferred model for main outcome
sa_main = run_sunab_style(df, "inefficient_pricing")
event_study_results = sa_main["event_df"].copy()

print("\nSun-Abraham-style event study: inefficient pricing")
display(event_study_results.round(4))

print("\nJoint test of pre-treatment leads (-4, -3, -2):")
print(sa_main["lead_test"])

print("\nOverall post-treatment ATT")
print("ATT =", round(sa_main["overall_att"], 4))
print("SE  =", round(sa_main["overall_se"], 4))
print("95% CI =", tuple(round(x, 4) for x in sa_main["overall_ci"]))

# Plot
plt.axhline(0, color="gray", linestyle="--")
plt.axvline(-1, color="gray", linestyle="--")
plt.errorbar(
    event_study_results["event_time"],
    event_study_results["coef"],
    yerr=1.96 * event_study_results["se"],
    fmt="o-",
    capsize=4
)
plt.title("Sun-Abraham-style event study: inefficient pricing")
plt.xlabel("Event time (reference = -1)")
plt.ylabel("ATT relative to year -1")
plt.show()

# Optional: other outcomes
sa_summary_rows = []
for y in ["inefficient_pricing", "avg_congestion", "avg_loss", "avg_total_lmp", "avg_energy"]:
    res = run_sunab_style(df, y)
    sa_summary_rows.append({
        "outcome": y,
        "overall_post_att": res["overall_att"],
        "overall_post_se": res["overall_se"],
        "ci_low": res["overall_ci"][0],
        "ci_high": res["overall_ci"][1]
    })

sunab_outcome_summary = pd.DataFrame(sa_summary_rows)
print("\nSun-Abraham-style overall post-treatment effects")
display(sunab_outcome_summary.round(4))

# ---------------------------------------------------------
# 8. Export outputs
# ---------------------------------------------------------
OUTDIR = PROJECT_ROOT / "output"
OUTDIR.mkdir(parents=True, exist_ok=True)

balance_table.to_csv(OUTDIR / "balance_table_pre.csv", index=False)
balance_ttests.to_csv(OUTDIR / "balance_ttests_pre.csv", index=False)
did_outcome_results.to_csv(OUTDIR / "did_outcome_results.csv", index=False)
event_study_results.to_csv(OUTDIR / "event_study_results.csv", index=False)
sunab_outcome_summary.to_csv(OUTDIR / "sunab_outcome_summary.csv", index=False)

print("\nSaved:")
print("-", OUTDIR / "balance_table_pre.csv")
print("-", OUTDIR / "balance_ttests_pre.csv")
print("-", OUTDIR / "did_outcome_results.csv")
print("-", OUTDIR / "event_study_results.csv")
print("-", OUTDIR / "sunab_outcome_summary.csv")